# Earthquake LSTM Forecasting and Evaluation

This notebook continues after `dgt5_earthquake_trainmodel.ipynb`. It loads the saved LSTM model and 12-month windows, forecasts validation and test months, evaluates MAE/RMSE/sMAPE, and compares train/validation/test errors to check for overfitting signals.

## Setup

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display


def resolve_forcoach_dir() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "dgt5_earthquake_trainmodel.ipynb").exists():
        return cwd
    if (cwd / "forcoach" / "dgt5_earthquake_trainmodel.ipynb").exists():
        return cwd / "forcoach"
    raise FileNotFoundError(
        "Could not find forcoach/dgt5_earthquake_trainmodel.ipynb. "
        "Start this notebook from the workspace root or from the forcoach folder."
    )


FORCOACH_DIR = resolve_forcoach_dir()
OUTPUT_DIR = FORCOACH_DIR / "data" / "model_outputs"
MODEL_DIR = FORCOACH_DIR / "models"

WINDOWS_PATH = OUTPUT_DIR / "step_13_lstm_windows.npz"
METADATA_PATH = OUTPUT_DIR / "step_13_window_metadata.csv"
HISTORY_PATH = OUTPUT_DIR / "step_14_training_history.csv"
MODEL_PATH = MODEL_DIR / "earthquake_lstm.keras"

FORECAST_TRAIN_PATH = OUTPUT_DIR / "step_15_train_forecast.csv"
FORECAST_VAL_PATH = OUTPUT_DIR / "step_15_validation_forecast.csv"
FORECAST_TEST_PATH = OUTPUT_DIR / "step_15_test_forecast.csv"
METRICS_PATH = OUTPUT_DIR / "step_15_evaluation_metrics.csv"
DIAGNOSTICS_PATH = OUTPUT_DIR / "step_15_overfitting_diagnostics.csv"

print(f"Forcoach directory: {FORCOACH_DIR}")
print(f"Model outputs: {OUTPUT_DIR}")

## Load Training Artifacts

In [ ]:
required_paths = {
    "window arrays": WINDOWS_PATH,
    "window metadata": METADATA_PATH,
    "training history": HISTORY_PATH,
    "trained model": MODEL_PATH,
}
missing = [f"{name}: {path}" for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing trained-model artifacts. Run all cells in "
        "forcoach/dgt5_earthquake_trainmodel.ipynb first. Missing:\n"
        + "\n".join(missing)
    )

windows = np.load(WINDOWS_PATH, allow_pickle=True)
X_train = windows["X_train"]
y_train = windows["y_train"]
X_val = windows["X_val"]
y_val = windows["y_val"]
X_test = windows["X_test"]
y_test = windows["y_test"]
feature_names = [str(name) for name in windows["feature_names"]]

metadata = pd.read_csv(METADATA_PATH)
for date_column in ["input_start_month", "input_end_month", "target_month"]:
    metadata[date_column] = pd.to_datetime(metadata[date_column], errors="raise")

history_df = pd.read_csv(HISTORY_PATH)
model = tf.keras.models.load_model(MODEL_PATH)

expected_input_shape = (X_train.shape[1], X_train.shape[2])
if model.input_shape[-2:] != expected_input_shape:
    raise AssertionError(
        f"Model input shape {model.input_shape[-2:]} does not match windows {expected_input_shape}."
    )

artifact_summary = pd.DataFrame(
    [
        {"split": "train", "X_shape": str(X_train.shape), "y_shape": str(y_train.shape)},
        {"split": "validation", "X_shape": str(X_val.shape), "y_shape": str(y_val.shape)},
        {"split": "test", "X_shape": str(X_test.shape), "y_shape": str(y_test.shape)},
    ]
)
print(f"Loaded model: {MODEL_PATH}")
print(f"Loaded {len(feature_names)} features.")
display(artifact_summary)
display(metadata.groupby("split")["target_month"].agg(["count", "min", "max"]))
display(history_df.tail())

## Forecast Validation and Test Data

In [ ]:
# section forecast validation and test data

## Evaluate MAE, RMSE, and sMAPE

In [ ]:
# section evaluate MAE, RMSE, sMAPE

## Overfitting Diagnostics

In [ ]:
# section: Overfitting diagnostics



diagnostics_df = pd.DataFrame(
    [
        {
            "check": "validation_mae_to_train_mae_ratio",
            "value": val_train_ratio,
            "status": ratio_status(val_train_ratio),
            "meaning": "Validation error compared with training error. Closer to 1 is better.",
        },
        {
            "check": "test_mae_to_train_mae_ratio",
            "value": test_train_ratio,
            "status": ratio_status(test_train_ratio),
            "meaning": "Test error compared with training error. Closer to 1 is better.",
        },
        {
            "check": "final_val_loss_to_final_train_loss_ratio",
            "value": final_val_to_train_loss_ratio,
            "status": ratio_status(final_val_to_train_loss_ratio),
            "meaning": "Training history gap at the final epoch.",
        },
        {
            "check": "final_val_loss_vs_best_val_loss_percent",
            "value": final_val_vs_best_val_percent,
            "status": "stable" if np.isfinite(final_val_vs_best_val_percent) and final_val_vs_best_val_percent <= 10 else "review history curve",
            "meaning": "Shows whether validation loss drifted up after its best epoch.",
        },
    ]
)

history_summary_df = pd.DataFrame(
    [
        {
            "epochs_completed": len(history_df),
            "best_validation_epoch": best_epoch,
            "best_validation_loss": best_val_loss,
            "final_training_loss": final_train_loss,
            "final_validation_loss": final_val_loss,
        }
    ]
)

display(history_summary_df)
display(diagnostics_df)

print(
    "Use these diagnostics as evidence, not proof: small train/validation/test gaps and "
    "stable validation loss support no clear overfitting signal; large gaps require review."
)

## Save Forecast and Evaluation Outputs

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_forecast_df.to_csv(FORECAST_TRAIN_PATH, index=False)
val_forecast_df.to_csv(FORECAST_VAL_PATH, index=False)
test_forecast_df.to_csv(FORECAST_TEST_PATH, index=False)
metrics_df.to_csv(METRICS_PATH, index=False)
diagnostics_df.to_csv(DIAGNOSTICS_PATH, index=False)

saved_outputs_df = pd.DataFrame(
    [
        {"artifact": "train forecasts", "path": str(FORECAST_TRAIN_PATH), "rows": len(train_forecast_df)},
        {"artifact": "validation forecasts", "path": str(FORECAST_VAL_PATH), "rows": len(val_forecast_df)},
        {"artifact": "test forecasts", "path": str(FORECAST_TEST_PATH), "rows": len(test_forecast_df)},
        {"artifact": "evaluation metrics", "path": str(METRICS_PATH), "rows": len(metrics_df)},
        {"artifact": "overfitting diagnostics", "path": str(DIAGNOSTICS_PATH), "rows": len(diagnostics_df)},
    ]
)

display(saved_outputs_df)
display(metrics_df[["split", "samples", "MAE", "RMSE", "sMAPE_percent", "bias_mean_error"]])